Step: Building Integrated Healthcare AI Agent

This notebook integrates three components:

1. Health analytics model (risk prediction)
2. Healthcare database
3. Medical knowledge RAG system

The agent predicts health risks and retrieves
trusted medical guidelines to generate explanations.

In [ ]:
from pathlib import Path

REPO_ROOT = Path.cwd().resolve()
MODELS_DIR = (REPO_ROOT / "models").resolve()
DATA_DIR = (REPO_ROOT / "data").resolve()

MODELS_DIR.mkdir(parents=True, exist_ok=True)
DATA_DIR.mkdir(parents=True, exist_ok=True)

print("Models dir:", MODELS_DIR)
print("Data dir:", DATA_DIR)

Step 1: Installing Required Libraries
We install libraries required for database access, machine learning inference,
and interaction with the medical knowledge RAG system.

In [ ]:
# Local setup (run once in a terminal, not inside the notebook):
# pip install -r ../requirements.txt
#
# This cell checks that the core agent dependencies are available.
import importlib.util

modules = [
    "langchain",
    "langchain_community",
    "langchain_chroma",
    "chromadb",
    "sentence_transformers",
    "langchain_groq",
    "pandas",
    "sklearn",
    "joblib",
    "dotenv",
]

missing = [m for m in modules if importlib.util.find_spec(m) is None]
if missing:
    raise ImportError(
        "Missing required packages: " + ", ".join(missing) + "\n" +
        "Install dependencies with: pip install -r ../requirements.txt"
    )

print("✅ Agent dependencies look installed")

Step 2: Import Required Libraries
This step loads all libraries required for the Healthcare AI Agent.

In [ ]:
import os
import joblib
import sqlite3
import pandas as pd

from langchain_groq import ChatGroq
from langchain_chroma import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings

Step 3: Loading Trained Health Risk Prediction Model

We load the trained machine learning model that predicts
the probability of diabetes risk based on health metrics.

In [ ]:
import joblib

model_path = MODELS_DIR / "health_risk_model.pkl"
scaler_path = MODELS_DIR / "scaler.pkl"

if not model_path.exists() or not scaler_path.exists():
    raise FileNotFoundError(
        f"Model or scaler not found in {MODELS_DIR}. "
        "Run the training notebook (health_analytics) to generate them first."
    )

model = joblib.load(model_path)
scaler = joblib.load(scaler_path)

print("Health risk model loaded successfully")

Step 4: Connecting to Healthcare Database

The database stores user health metrics, medications,
and wellness goals. This allows the AI agent to track
patient health data over time.

In [ ]:
db_path = "../data/health.db"

conn = sqlite3.connect(db_path)

print("Database connected")

In [ ]:
I'm ready to assist. However, you haven't provided the medical guidelines yet. Please provide the guidelines, and I'll be happy to help with the patient's condition and suggest monitoring or prevention advice based on the provided information.

In [ ]:
# Initialize Retriever (Added fix)
from langchain_chroma import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)
db_path = "../data/vector_db"
vector_store = Chroma(
    persist_directory=db_path,
    embedding_function=embeddings,
)
retriever = vector_store.as_retriever(search_kwargs={"k": 3})


In [ ]:
def get_medical_guidance(query):

    docs = retriever.invoke(query)

    context = "\n\n".join([doc.page_content for doc in docs])

    return context

Step 5: Predicting Health Risk

This function uses the trained machine learning model
to predict diabetes risk from patient health metrics.

In [ ]:
def predict_diabetes(pregnancies, glucose, blood_pressure,
                     skin_thickness, insulin, bmi,
                     diabetes_pedigree, age):

    import pandas as pd

    data = pd.DataFrame([{
        "Pregnancies": pregnancies,
        "Glucose": glucose,
        "BloodPressure": blood_pressure,
        "SkinThickness": skin_thickness,
        "Insulin": insulin,
        "BMI": bmi,
        "DiabetesPedigreeFunction": diabetes_pedigree,
        "Age": age
    }])

    # Scale using the globally loaded scaler
    data_scaled = scaler.transform(data)

    prediction = model.predict(data_scaled)[0]

    if prediction == 1:
        return "High Diabetes Risk"
    else:
        return "Low Diabetes Risk"


Step 6: Initializing the AI Medical Assistant

We initialize the Groq LLM which will generate
AI explanations using retrieved medical guidelines.

In [ ]:
from dotenv import load_dotenv
load_dotenv()

groq_key = os.getenv("GROQ_API_KEY")

llm = ChatGroq(
    api_key=groq_key,
    model="llama-3.1-8b-instant",
    temperature=0
)

print("AI Medical Assistant ready")

Step 7: Building the Healthcare AI Agent

This step combines:

• ML health risk prediction
• Medical guideline retrieval (RAG)
• AI explanation generation

to provide personalized healthcare insights.

In [ ]:
def healthcare_ai_agent(patient_data):

    risk = predict_diabetes(**patient_data)

    medical_context = get_medical_guidance(
        "diabetes treatment and prevention"
    )

    prompt = f"""
You are a professional healthcare AI assistant.

Rules:
1. Use ONLY the provided medical guidelines.
2. Do NOT add external medical knowledge.
3. If information is not present in the guidelines, say:
   "The available medical guidelines do not provide enough information."

Patient Health Risk Prediction:
{risk}

Using the medical guidelines below,
explain the patient's condition and suggest monitoring or prevention advice.

Medical Guidelines:
{medical_context}
"""

    response = llm.invoke(prompt)

    return response.content

Step 8: Testing the Integrated Healthcare AI Agent

We provide patient health metrics and observe the AI agent's
risk prediction and medical guidance.

In [ ]:
patient = {
"pregnancies":2,
"glucose":140,
"blood_pressure":85,
"skin_thickness":20,
"insulin":85,
"bmi":30,
"diabetes_pedigree":0.5,
"age":45
}


In [ ]:
healthcare_ai_agent(patient)